In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

import numpy as np

Load the datasets

In [2]:
data=np.load("data/digit_splits.npz")

# Get training data
X_train = data["X_train"]
y_train = data["y_train"]

# Get validation data
X_val = data["X_val"]
y_val = data["y_val"]

# Get testing data (keep aside for evaluation)
X_test = data["X_test"]
y_test = data["y_test"]

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)



Training data: (1257, 8, 8)
Validation data: (270, 8, 8)
Testing data: (270, 8, 8)


In [3]:
# Convert images into PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)

# Convert labels into integer tensors
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

# Add channel dimension to images
X_train = X_train.unsqueeze(1)
X_val = X_val.unsqueeze(1)

print("CNN training input shape:", X_train.shape)
print("CNN validation input shape:", X_val.shape)


CNN training input shape: torch.Size([1257, 1, 8, 8])
CNN validation input shape: torch.Size([270, 1, 8, 8])


In [4]:

# Define our CNN model

class DigitCNN(nn.Module):

    def __init__(self):
        super(DigitCNN, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(kernel_size=2)

        # Fully connected layers
        self.fc1 = nn.Linear(16 * 4 * 4, 64)

        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        # Feature extraction
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        # Flatten the feature maps
        x = x.view(x.size(0), -1)

        # Classification
        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)

        return x


In [5]:

# Create CNN model
model = DigitCNN()

# Test with one batch of images
sample_input = X_train[:4]

# Get predictions
output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([4, 1, 8, 8])
Output shape: torch.Size([4, 10])


In [6]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

Train the model

In [7]:
epochs = 30

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_train)

    loss = criterion(outputs, y_train)

    loss.backward()

    optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/30], Loss: 2.3087
Epoch [2/30], Loss: 2.2929
Epoch [3/30], Loss: 2.2788
Epoch [4/30], Loss: 2.2654
Epoch [5/30], Loss: 2.2524
Epoch [6/30], Loss: 2.2397
Epoch [7/30], Loss: 2.2267
Epoch [8/30], Loss: 2.2134
Epoch [9/30], Loss: 2.1998
Epoch [10/30], Loss: 2.1858
Epoch [11/30], Loss: 2.1710
Epoch [12/30], Loss: 2.1555
Epoch [13/30], Loss: 2.1394
Epoch [14/30], Loss: 2.1227
Epoch [15/30], Loss: 2.1054
Epoch [16/30], Loss: 2.0875
Epoch [17/30], Loss: 2.0688
Epoch [18/30], Loss: 2.0494
Epoch [19/30], Loss: 2.0294
Epoch [20/30], Loss: 2.0088
Epoch [21/30], Loss: 1.9875
Epoch [22/30], Loss: 1.9654
Epoch [23/30], Loss: 1.9426
Epoch [24/30], Loss: 1.9189
Epoch [25/30], Loss: 1.8946
Epoch [26/30], Loss: 1.8695
Epoch [27/30], Loss: 1.8436
Epoch [28/30], Loss: 1.8171
Epoch [29/30], Loss: 1.7898
Epoch [30/30], Loss: 1.7619


Evalution on validation set

In [8]:
# Validation
model.eval()

with torch.no_grad():

    val_outputs = model(X_val)

    predictions = torch.argmax(val_outputs, dim=1)

    correct = (predictions == y_val).sum().item()

    accuracy = correct / len(y_val) * 100

    print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 79.26%


Training dataset in batches

In [9]:
train_dataset=TensorDataset(X_train, y_train)

train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)

Validation dataset in batches

In [10]:
val_dataset=TensorDataset(X_val, y_val)

val_loader=DataLoader(val_dataset, batch_size=32, shuffle=False)

Train the model in batches for better performance and memory efficiency.

In [11]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/30], Loss: 1.3117
Epoch [2/30], Loss: 0.6927
Epoch [3/30], Loss: 0.4502
Epoch [4/30], Loss: 0.3273
Epoch [5/30], Loss: 0.2522
Epoch [6/30], Loss: 0.2128
Epoch [7/30], Loss: 0.1823
Epoch [8/30], Loss: 0.1594
Epoch [9/30], Loss: 0.1473
Epoch [10/30], Loss: 0.1314
Epoch [11/30], Loss: 0.1137
Epoch [12/30], Loss: 0.1064
Epoch [13/30], Loss: 0.0960
Epoch [14/30], Loss: 0.0835
Epoch [15/30], Loss: 0.0789
Epoch [16/30], Loss: 0.0706
Epoch [17/30], Loss: 0.0633
Epoch [18/30], Loss: 0.0641
Epoch [19/30], Loss: 0.0627
Epoch [20/30], Loss: 0.0527
Epoch [21/30], Loss: 0.0516
Epoch [22/30], Loss: 0.0450
Epoch [23/30], Loss: 0.0441
Epoch [24/30], Loss: 0.0439
Epoch [25/30], Loss: 0.0398
Epoch [26/30], Loss: 0.0327
Epoch [27/30], Loss: 0.0301
Epoch [28/30], Loss: 0.0285
Epoch [29/30], Loss: 0.0274
Epoch [30/30], Loss: 0.0254


Validation in batches

In [12]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

accuracy = (correct / total) * 100

print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 97.04%


In [14]:
from pathlib import Path

# Create models folder
Path("models").mkdir(exist_ok=True)

# Save trained model
torch.save(model.state_dict(), "models/cnn_model.pth")

print("CNN model saved successfully!")

CNN model saved successfully!
